# Model Evaluation Methodology

> The system is now faster: the model is quantized, the engine and kernels have changed, and decoding parameters are fixed. Before deployment, one question remains: are the outputs still correct? Every optimization can quietly reduce quality while tokens/s rises.
>
> This chapter explains how to answer two questions credibly: “Did quality regress?” and “Is A really better than B?” We cover the evaluation pipeline, evaluation targets, benchmarks and metrics, LLM-as-Judge bias, confidence intervals, a minimal implementation, tool selection, and a pre-deployment checklist.


## 1. The Evaluation Pipeline

Evaluation is a pipeline, and every stage affects the final score:

```text
Dataset / Benchmark
        ↓
Prompt / Chat Template
        ↓
Generation Configuration
        ↓
Model / Engine
        ↓
Parser
        ↓
Metric or Judge
        ↓
Aggregation + Confidence Interval
```

For “A beats B by one point” to be meaningful, only the evaluated model may change. A prompt, parser, or temperature change can exceed the model difference.


## 2. Classifying Evaluation Targets

“Evaluation” covers at least four different questions:

- **Model quality**: knowledge, mathematics, code, and instruction following.
- **Inference quality regression**: whether quantization, kernels, or engines degrade outputs.
- **Serving performance**: TTFT, TPOT, throughput, and memory.
- **System cost**: cost per million Tokens, GPU count, and power.

Quality and performance belong together because optimization is a trade-off, but their measurements must remain distinct.


In [ ]:
configs = {
    "BF16": {"memory_gb":14.0, "quality":72.4, "throughput":1.0},
    "INT8": {"memory_gb":7.2,  "quality":72.2, "throughput":1.35},
    "INT4": {"memory_gb":3.8,  "quality":71.3, "throughput":1.75},
}
for name,v in configs.items():
    print(name, v)
print("The real question is not whether INT4 is fast, but whether saved memory and throughput justify the quality loss.")


The question is not merely whether INT4 is faster, but whether its memory and throughput gains justify a 1.1-point quality loss. That decision requires trustworthy quality scores from this chapter and trustworthy performance measurements from the inference-systems chapter.


## 3. Benchmarks and Metrics

A **Benchmark** defines the questions; a **Metric** defines scoring. MMLU contains multiple-choice questions, GSM8K contains grade-school word problems, and HumanEval contains programming tasks. Accuracy counts correct choices, exact match requires an exact answer, and pass@k succeeds when any of $k$ generated programs passes tests.

| Benchmark | Capability | Typical Metric |
|:---|:---|:---|
| MMLU / GPQA | knowledge and reasoning | accuracy |
| GSM8K / MATH | mathematics | parsed exact match |
| HumanEval / LiveCodeBench | code | pass@k with execution |
| SWE-bench | repository-level fixes | tests passed |
| Open dialogue | subjective quality | pairwise preference / LLM-as-Judge |

Always ask whether a benchmark measures the capability your application needs. Scores are comparable only with matching prompts, few-shot settings, parsers, and generation configuration.


### What Common Benchmark Questions Look Like

- **MMLU**: English four-choice questions over 57 subjects, scored by accuracy.
- **C-Eval**: Chinese four-choice questions over 52 academic and professional subjects.
- **CMMLU**: Chinese-context multiple-choice knowledge questions.
- **GSM8K**: English grade-school word problems scored by the final numeric answer.

The next cell prints representative question text, choices, targets, and scoring rules.


In [ ]:
benchmarks = {
    "MMLU (English multiple choice)": {
        "subject": "high-school physics",
        "question": "A 2 kg object accelerates at 3 m/s^2. What is the net force?",
        "choices": ["1.5 N", "6 N", "5 N", "0.67 N"],
        "answer": "B",
        "metric": "accuracy by answer letter",
    },
    "C-Eval (Chinese multiple choice)": {
        "subject": "high-school geography",
        "question": "What is the basic pattern of China’s terrain?",
        "choices": ["high east, low west", "high west, low east in terraces", "high south, low north", "high edges, low center"],
        "answer": "B",
        "metric": "accuracy by answer letter",
    },
    "CMMLU (Chinese-context knowledge)": {
        "subject": "middle-school language",
        "question": "Which meaning is closest to the idiom about mending the pen after losing a sheep?",
        "choices": ["too late to help", "fixing a problem afterward can prevent further loss",
                    "acting extremely cautiously", "having an impressive appearance without substance"],
        "answer": "B",
        "metric": "accuracy by answer letter",
    },
    "GSM8K (math word problems)": {
        "subject": "elementary mathematics",
        "question": "Natalia sold clips to 48 friends in April, and half as many in May. "
                    "How many clips did she sell altogether?",
        "choices": None,
        "answer": "72",
        "metric": "exact match after parsing the final number",
    },
}

for name, item in benchmarks.items():
    print(f"=== {name}（{item['subject']}）")
    print("Question:", item["question"])
    if item["choices"]:
        for letter, c in zip("ABCD", item["choices"]):
            print(f"  {letter}. {c}")
    print("Reference:", item["answer"], " metric:", item["metric"])
    print()


Three details matter. First, multiple-choice evaluation commonly compares option log-likelihoods instead of generating text, which is more reproducible. Second, Chinese benchmarks are independently authored rather than translations, so Chinese applications require Chinese evaluation. Third, GSM8K parsers extract a final number after reasoning; a parsing failure is not the same as a reasoning failure. Code benchmarks instead execute unit tests, making their metric behavioral rather than textual.


## 4. Bias in LLM-as-Judge

Open-ended summaries and conversations cannot be scored by simple equality. A strong model can compare two answers, but judges exhibit position, length, style, and self-preference biases.

Mitigations include a fixed rubric, **dual-order judging** (judge A/B and B/A), treating inconsistent decisions as ties or sending them to human review, and retaining raw judge outputs for audit. The next simulation measures how often position bias flips a decision.


In [ ]:
import random

def biased_judge(golden_position, bias=0.15):
    """Simulated judge: follow true quality 85% of the time and unconditionally favor position one 15%."""
    if random.random() < bias:
        return 1
    return golden_position

random.seed(42)
pairs = 200
flips = 0
for _ in range(pairs):
    golden = random.choice([1, 2])          # randomize which side holds the better answer
    picked1 = biased_judge(golden)          # first judgment: position equals answer ID
    picked2 = 3 - biased_judge(3 - golden)  # after swapping, convert position one back to answer two
    if picked1 != picked2:
        flips += 1

print(f"Across {pairs} answer pairs, swapping order flips {flips} decisions ({flips / pairs:.0%})")
print()
print('Conclusion')
print('Conclusion')


In [ ]:
# Compare agreement with flips: judge conclusions are less stable than they seem
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 3))
plt.bar(["consistent", "flipped"], [pairs - flips, flips],
        color=["tab:green", "tab:red"])
plt.ylabel("pairs")
plt.title(f"Position bias flips {flips / pairs:.0%} of verdicts")
plt.show()


A judge with only 15% preference for position 1 flips 15% of pairwise conclusions. Bias does not disappear through averaging; it directly creates unreliable decisions. Dual-order judging is the lowest-cost way to expose these flips.


## 5. Confidence Intervals

Sampling creates uncertainty. On only 20 questions, 15 correct (75%) versus 14 correct (70%) does not establish superiority. Bootstrap resampling repeatedly draws answer records with replacement, computes a score each time, and uses the resulting distribution as a confidence interval.


In [ ]:
import random, statistics

random.seed(42)
scores = [1]*15 + [0]*5

boots=[]
for _ in range(5000):
    sample=[random.choice(scores) for _ in scores]
    boots.append(sum(sample)/len(sample))

boots.sort()
print("accuracy:", sum(scores)/len(scores))
print("95% bootstrap interval:", round(boots[125],3), round(boots[-126],3))


An interval of [0.55, 0.90] means that 20 questions support a very wide range of plausible performance. When model intervals overlap substantially, a few score points are insufficient evidence: increase the sample size or report that the experiment cannot distinguish them. A leaderboard difference such as 76.1 versus 76.4 should immediately prompt the question: what are the intervals?


## 6. Building a Minimal Evaluation Pipeline

We now implement dataset → model answer → parsing → metric → confidence interval without a framework. A mock model answers correctly with configurable probability and varies its output format, exposing parser failures. In a real system, replace `mock_model` with an OpenAI-compatible endpoint while leaving the rest of the pipeline unchanged.


In [ ]:
import random
import re

random.seed(42)

# 1) Dataset: twelve tiny questions with unique numeric answers; real evaluation uses standard benchmarks
questions = [
    ("Sam has 3 apples and buys 5 more. How many total?", 8),
    ("A car travels 60 km per hour. How far in 2 hours?", 120),
    ("A dozen eggs contains 12. How many in two dozen?", 24),
    ("A $100 shirt is discounted to 80%. What is the price?", 80),
    ("A class has 30 students, half girls. How many girls?", 15),
    ("What is the sum from 1 through 10?", 55),
    ("A 9-meter rope loses 4 meters. How much remains?", 5),
    ("A movie starts at 7 and lasts 2 hours. When does it end?", 9),
    ("A 240-page book is read at 80 pages daily. How many days?", 3),
    ("A case has 24 bottles; 10 are consumed. How many remain?", 14),
    ("What is 3 times 7?", 21),
    ("Temperature rises 8 degrees from 5. What is it now?", 13),
]

# 2) Mock model under evaluation: adjustable accuracy and randomized output format
def mock_model(question, answer, correctness=0.7):
    """Answer correctly with probability correctness; otherwise return a nearby wrong number."""
    correct = random.random() < correctness
    value = answer if correct else answer + random.choice([1, 2, 3])
    style = random.choice(["plain", "cn", "en"])
    if style == "plain":
        return str(value)
    if style == "cn":
        return f"The answer is {value}."
    return f"The answer is: {value}."

# 3) Parser: extract the final integer from model output
def parse_answer(text):
    numbers = re.findall(r"\d+", text)
    return int(numbers[-1]) if numbers else None

for text in [mock_model(questions[0][0], questions[0][1]) for _ in range(3)]:
    print(f"Output: {text!r:<28} -> parsed: {parse_answer(text)}")
print("Key observation: the parser must accept all three output formats from the same model.")


The mock produces plain numbers and sentence-form answers, reflecting real variation in model output. The parser must handle all formats. We next compare two mock models and attach bootstrap intervals to both scores.


In [ ]:
# 4) Run two models and calculate accuracy plus bootstrap confidence intervals
def run_eval(model_fn):
    results = []
    for q, gold in questions:
        pred = parse_answer(model_fn(q, gold))
        results.append(1 if pred == gold else 0)
    return results

def bootstrap_ci(results, n_boot=5000):
    stats = []
    for _ in range(n_boot):
        sample = random.choices(results, k=len(results))
        stats.append(sum(sample) / len(sample))
    stats.sort()
    return stats[int(0.025 * n_boot)], stats[int(0.975 * n_boot)]

random.seed(11)
scores_a = run_eval(lambda q, a: mock_model(q, a, correctness=0.75))
scores_b = run_eval(lambda q, a: mock_model(q, a, correctness=0.60))

acc_a, acc_b = sum(scores_a) / len(scores_a), sum(scores_b) / len(scores_b)
lo_a, hi_a = bootstrap_ci(scores_a)
lo_b, hi_b = bootstrap_ci(scores_b)

print(f"Model A: accuracy {acc_a:.2f}  95% CI [{lo_a:.2f}, {hi_a:.2f}]")
print(f"Model B: accuracy {acc_b:.2f}  95% CI [{lo_b:.2f}, {hi_b:.2f}]")
print()
print("Key observation: A appears higher, but the two confidence intervals overlap;")
print("With only twelve questions, this gap does not prove that A is truly stronger.")


In [ ]:
# Comparison with error bars: overlapping intervals mean the gap is not significant
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.2))
for i, (name, acc, lo, hi) in enumerate([
        ("model A", acc_a, lo_a, hi_a), ("model B", acc_b, lo_b, hi_b)]):
    plt.errorbar(i, acc, yerr=[[acc - lo], [hi - acc]], fmt="o", capsize=6)
plt.xticks([0, 1], ["model A", "model B"])
plt.ylabel("accuracy (with 95% CI)")
plt.ylim(0, 1.05)
plt.title("Overlapping CIs: the gap is not conclusive at n=12")
plt.show()


Model A appears 17 points ahead, but the intervals overlap heavily. Twelve questions cannot reliably resolve a difference of this size. “The measurement is inconclusive” is more useful than a misleading 0.75 versus 0.58.


## 7. Registering a Custom Evaluation in lm-evaluation-harness

In lm-eval, each benchmark is a YAML task describing where data comes from, how prompts and choices are built, where the target is stored, and which metric is used. Place the configuration in a directory and pass `--include_path` to register it alongside built-in tasks.

| Field | Pipeline Stage | Purpose |
|:---|:---|:---|
| `dataset_path`, `dataset_kwargs` | Dataset | local JSONL or Hugging Face data |
| `process_docs` | Dataset | filtering and preprocessing |
| `doc_to_text` | Prompt | Jinja2 prompt template |
| `doc_to_choice` | Choices | option text for log-likelihood |
| `doc_to_target` | Target | correct option index |
| `output_type`, `metric_list` | Metric | task and scoring method |

The example creates Chinese idiom and common-knowledge subtasks.


In [ ]:
import json, os

# 1) Question bank: twelve four-choice questions, with task distinguishing two subtasks
items = [
    ("idiom", "What does 'adding feet to a snake' criticize?",
     ["ruining something with unnecessary work", "excellent skill", "strength in numbers", "preparing early"], 0),
    ("idiom", "What does 'waiting by a stump for a rabbit' criticize?",
     ["expecting gain without effort", "persistence", "adaptability", "frugality"], 0),
    ("idiom", "What does 'when the water falls, the rocks appear' mean?",
     ["the truth becomes clear", "a flood", "hard stones", "a wide water surface"], 0),
    ("idiom", "Whom does 'a frog at the bottom of a well' describe?",
     ["a narrow-minded person", "a brave person", "a wealthy person", "a strong swimmer"], 0),
    ("idiom", "What does 'sending charcoal in snowy weather' mean?",
     ["helping someone in urgent need", "cold weather", "pretending kindness", "wasting resources"], 0),
    ("idiom", "Whom does 'playing music to a cow' criticize?",
     ["someone speaking without regard for the audience", "a poor musician", "the cow", "a composer"], 0),
    ("common", "What is the capital of China?", ["Beijing", "Shanghai", "Guangzhou", "Chengdu"], 0),
    ("common", "How many months are in one year?", ["12", "10", "24", "6"], 0),
    ("common", "What is the boiling point of water at standard atmospheric pressure?",
     ["100 degrees Celsius", "90 degrees Celsius", "80 degrees Celsius", "120 degrees Celsius"], 0),
    ("common", "Who wrote 'Quiet Night Thought'?", ["Li Bai", "Du Fu", "Bai Juyi", "Su Shi"], 0),
    ("common", "What does Earth's rotation produce?",
     ["day and night", "the seasons", "moon phases", "the main cause of tides"], 0),
    ("common", "What is normal human body temperature approximately?",
     ["37 degrees Celsius", "30 degrees Celsius", "42 degrees Celsius", "25 degrees Celsius"], 0),
]

os.makedirs("myzh_eval", exist_ok=True)
with open("myzh_eval/myzh.jsonl", "w") as f:
    for task, q, choices, label in items:
        f.write(json.dumps({"task": task, "question": q,
                            "choices": choices, "label": label},
                           ensure_ascii=False) + "\n")

# 2) process_docs: filter subtasks from the mixed question bank using task
with open("myzh_eval/utils.py", "w") as f:
    f.write(
        "def only_idiom(ds):\n"
        "    return ds.filter(lambda d: d['task'] == 'idiom')\n\n"
        "def only_common(ds):\n"
        "    return ds.filter(lambda d: d['task'] == 'common')\n"
    )

# 3) Two task YAML files; only task name and process_docs differ
yaml_template = """output_type: multiple_choice
task: {task_name}
dataset_path: json
dataset_kwargs:
  data_files: myzh_eval/myzh.jsonl
test_split: train
process_docs: !function utils.{fn}
doc_to_text: "Question: {{{{question}}}}\nA. {{{{choices[0]}}}}\nB. {{{{choices[1]}}}}\nC. {{{{choices[2]}}}}\nD. {{{{choices[3]}}}}\nAnswer: "
doc_to_choice: "{{{{choices}}}}"
doc_to_target: "{{{{label}}}}"
metric_list:
  - metric: acc
"""
for task_name, fn in [("myzh_idiom", "only_idiom"), ("myzh_common", "only_common")]:
    with open(f"myzh_eval/{task_name}.yaml", "w") as f:
        f.write(yaml_template.format(task_name=task_name, fn=fn))

print(open("myzh_eval/myzh_idiom.yaml").read())
print("Question bank and two task configs written to myzh_eval/.")


Three YAML details deserve attention. `output_type: multiple_choice` selects reproducible option log-likelihood scoring instead of generation and parsing. `doc_to_text` is the exact Jinja2 prompt template and must remain fixed across comparisons. `process_docs: !function utils.only_idiom` calls local preprocessing, which can filter, deduplicate, or stratify real datasets. We next register the directory and verify both tasks are discoverable.


In [ ]:
from lm_eval.tasks import TaskManager

# include_path is the registration mechanism: give TaskManager the directory and YAML task names become available
tm = TaskManager(include_path="myzh_eval")

mine = sorted(t for t in tm.all_tasks if t.startswith("myzh"))
print("Registered custom tasks:", mine)


Custom tasks and built-in tasks now use the same pipeline and scorer. The next example evaluates GPT-2 and Qwen2.5-0.5B-Instruct on both tasks. The equivalent command is:

```bash
lm_eval --model hf --model_args pretrained=Qwen/Qwen2.5-0.5B-Instruct \
  --include_path myzh_eval --tasks myzh_idiom,myzh_common --device cpu
```


In [ ]:
import torch
from lm_eval import evaluator

models = ["gpt2", "Qwen/Qwen2.5-0.5B-Instruct"]
scores = {}  # {model name: {task name: accuracy}}

for name in models:
    torch.manual_seed(42)
    result = evaluator.simple_evaluate(
        model="hf",
        model_args={"pretrained": name, "dtype": "float32", "device": "cpu"},
        tasks=["myzh_idiom", "myzh_common"],
        task_manager=tm,          # include our registration directory
        verbosity="ERROR",
    )
    scores[name] = {t: v["acc,none"] for t, v in result["results"].items()}
    print(name, {t: round(a, 3) for t, a in scores[name].items()})


Both models now have scores for both tasks. The final step creates a report-style grouped bar chart and adds a 25% random baseline for four-choice questions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

task_display = {"myzh_idiom": "Idiom", "myzh_common": "Common"}
model_display = {"gpt2": "GPT-2 124M", "Qwen/Qwen2.5-0.5B-Instruct": "Qwen2.5 0.5B"}
task_names = ["myzh_idiom", "myzh_common"]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(task_names))
width = 0.35
for i, name in enumerate(models):
    vals = [100 * scores[name][t] for t in task_names]
    bars = ax.bar(x + (i - 0.5) * width, vals, width,
                  label=model_display[name], edgecolor="black", linewidth=0.6)
    for b, v in zip(bars, vals):  # value labels above bars, standard in technical reports
        ax.text(b.get_x() + b.get_width() / 2, v + 1.2, f"{v:.0f}",
                ha="center", va="bottom", fontsize=10)

# Random-guess baseline for four choices: scores below it indicate negative knowledge
ax.axhline(25, color="gray", linestyle="--", linewidth=1)
ax.text(1.42, 26, "random = 25%", color="gray", fontsize=9, ha="right")

ax.set_xticks(x)
ax.set_xticklabels([task_display[t] for t in task_names])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Chinese Benchmarks: GPT-2 vs Qwen2.5-0.5B (12 items each)")
ax.set_ylim(0, 100)
ax.legend(loc="upper right", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


Two cautions follow from the chart. GPT-2's low Chinese scores reflect mismatched training data and tokenization, illustrating why benchmark relevance comes before ranking. Also, twelve questions have enormous variance, so this chart verifies only that the pipeline works. A real conclusion needs hundreds or thousands of examples. The complete workflow is now dataset → YAML registration → standardized scoring → report.


## 8. Evaluation Tool Map

- **lm-evaluation-harness / OpenCompass**: complete standard benchmark pipelines.
- **AlpacaEval / MT-Bench-style tools**: dialogue with LLM-as-Judge.
- **SWE-bench harness**: repository tasks with executable tests.
- **Promptfoo / DeepEval**: application regression tests suitable for CI.
- **vLLM / SGLang benchmarks**: serving performance, not model quality.

Tools change, but the pipeline remains. Choose by identifying which stages a tool fixes and which remain your responsibility.


### Evaluation Frameworks

**lm-evaluation-harness** provides hundreds of standard tasks and controls datasets, prompting, scoring, and aggregation. **OpenCompass** offers broad Chinese and English coverage. **EvalScope** integrates ModelScope models and inference backends for pre-deployment quality and load testing.

```bash
lm_eval --model hf \
  --model_args pretrained=Qwen/Qwen2.5-1.5B \
  --tasks mmlu --batch_size 8
```

Even when a framework fixes datasets and metrics, you must still control chat templates and generation settings. A changed template makes scores incomparable.


In [ ]:
tools = [
    ("lm-evaluation-harness", "EleutherAI", "general standard for English leaderboards, with broad tasks and common academic-report use"),
    ("OpenCompass", "Shanghai AI Lab", "broad Chinese and English coverage; engine behind the OpenCompass leaderboard"),
    ("EvalScope", "Alibaba ModelScope", "strong vLLM and MaaS integration; evaluation plus load testing; Chinese-friendly"),
    ("lighteval", "Hugging Face", "tight integration with the transformers and datasets ecosystem"),
    ("AlpacaEval / MT-Bench", "academia", "open-dialogue quality with LLM-as-Judge, including the bias discussed earlier"),
    ("Promptfoo / DeepEval", "application layer", "CI regression tests with custom assertions"),
]
print(f"{'Tool':<28}{'From':<14}Role")
print("-" * 80)
for name, org, use in tools:
    print(f"{name:<30}{org:<14}{use}")
print()
print("Key observation: the first four tools run standard benchmarks; the last two guard application-specific behavior,")
print("They complement each other: leaderboard scores prevent model-selection mistakes; business regressions prevent deployment incidents.")


## 9. Minimum Pre-Deployment Comparison

To compare BF16 with AWQ INT4, fix every stage except precision:

| Fixed Item | Content |
|:---|:---|
| Model and tokenization | revision, tokenizer, chat template |
| Questions | identical dataset |
| Generation | temperature=0 or fixed seed |
| Length budget | maximum context and output |
| Hardware and load | same GPUs and concurrency |

Record both sides:

| Measurement | Content |
|:---|:---|
| Quality | benchmark score + confidence interval |
| Memory | peak GPU memory |
| TTFT / TPOT | P50 / P95 |
| Throughput | Tokens/s |

Only then can you judge whether INT4's memory and throughput gains justify its quality difference.


## Summary

### What You Learned (by section order)

| # | Section | Core Content |
|:---|:---|:---|
| 1 | Evaluation Landscape | What 2025 papers/industry evaluate, evaluation evolution, minimal starter suite |
| 2 | Core Repos | Selection among lm-eval-harness, AlpacaEval, FastChat, DeepEval |
| 3 | OpenAI-Compatible API | `local-chat-completions` vs `local-completions`, connecting any compatible API |
| 4 | LLM-as-Judge | MT-Bench prompt + OpenAI SDK real scoring implementation |
| 5 | Results Aggregation & Visualization | Comparison table + radar chart + bar chart + win rate matrix + 5 composite score methods |
| 6 | AlpacaEval in Practice | Complete CLI + Python API pipeline, OpenAI-Compatible integration |
| 7 | Specialized Evaluation | RAG (RAGAS faithfulness/relevance), Code (pass@k vs stability) |
| 8 | LLM-as-Judge Bias & Consistency | Position/Length/Egocentric Bias + CR@K/Prompt Robustness |
| 9 | Metrics System | acc / exact_match / pass@k / win_rate / Elo, 2025 new metrics (AIME, SWE-bench, LiveCodeBench) |
| 10 | Common Pitfalls | Data level (contamination/prompt sensitivity/few-shot), engineering level (API confusion/seed/temperature), interpretation level |
| 11 | Quick Reference | CLI + Python API, adjustable to your environment before running |

### Recommended Repos

```bash
# Core evaluation frameworks
git clone https://github.com/EleutherAI/lm-evaluation-harness.git  # Common open-source evaluation framework, supports many tasks
git clone https://github.com/tatsu-lab/alpaca_eval.git              # LLM-as-Judge, 805 prompts
git clone https://github.com/lm-sys/FastChat.git                    # MT-Bench + Chatbot Arena

# Advanced tools
pip install deepeval       # CI/CD evaluation (hallucination detection, G-Eval, 40+ metrics)
pip install ragas          # RAG evaluation (faithfulness, relevance)
```

### Next Steps

```
Level 1 (do today):
  1. pip install lm-eval openai
  2. Run gsm8k --limit 50 with the DeepSeek API
  3. Understand every field in the output JSON

Level 2 (this month):
  1. Deploy an open-source model with vLLM
  2. Run 4 core datasets: gsm8k + mmlu + humaneval + ifeval
  3. Draw a radar chart + compute the geometric mean
  4. Verify scores against a public leaderboard

Level 3 (ongoing):
  1. Integrate AlpacaEval / MT-Bench to evaluate dialogue quality
  2. Build custom eval sets for your business scenario (RAGAS / custom prompts)
  3. Track consistency metrics (CR@K), not just accuracy
```

### Key Concepts Quick Reference

| Concept | One-Line Explanation |
|:---|:---|
| **loglikelihood vs generate_until** | MCQ computes probability (Completions API), generation uses autoregression (Chat API) |
| **pass@k and stability** | pass@k is optimistic (at least one of k correct); stability can be tracked with custom repeated pass / all-pass@k |
| **LC Win Rate** | Win rate after controlling for answer length, avoiding "longer wins" |
| **Geometric vs Arithmetic Mean** | Geometric mean penalizes weak spots; model imbalance shows immediately |
| **CR@K** | Ask the same question K times, the proportion of correct answers — measures stability |
| **Position / Length / Egocentric Bias** | The three major LLM-as-Judge biases, requiring randomization + LC + cross-validation |
| **Data Contamination** | Training set leaked eval questions causing inflated scores — use dynamic datasets like LiveCodeBench to prevent it |

Bottom line: deploying a model without evaluation carries high risk. Evaluation = standardized test questions + automated grading + reproducible scores + continuous monitoring.

## Exercises

1. **Perplexity by Hand**

   Given a 3-token sequence, the model's output logit probabilities (after softmax) are [0.5, 0.3, 0.2], [0.1, 0.7, 0.2], [0.4, 0.1, 0.5]. Calculate perplexity by hand.

   <details><summary>Hint</summary>Take the log probability for each token, average them, then exponentiate. Lower perplexity means the model is more "confident."</details>

2. **LLM-as-Judge Bias Detection**

   Design an experiment: for the same pair of answers, swap positions (answer_a and answer_b), repeat 3 times, and tally the difference in GPT-4's scores. Compute the Position Bias rate in code.

   <details><summary>Hint</summary>Position Bias rate = number of inconsistent scores before/after swapping / total count. Inconsistency means the judge was influenced by position.</details>

3. **Evaluation Report Writing**

   Using the simulated data below, draw a radar chart, compute the geometric mean score of both models across 4 benchmarks, and write your conclusions.

   ```python
   scores = {
       'Model A': {'gsm8k': 82, 'mmlu': 71, 'humaneval': 65, 'ifeval': 78},
       'Model B': {'gsm8k': 78, 'mmlu': 75, 'humaneval': 70, 'ifeval': 72},
   }
   ```

   <details><summary>Hint</summary>Geometric mean = (a * b * c * d) ** (1/4). Compare both models' geometric means and observe which model is more "balanced" on the radar chart.</details>

### Exercise 1: Implement a Bootstrap Confidence Interval

Resample with replacement, compute a mean for each sample, sort the results, and take the 2.5th and 97.5th percentiles.

Hint: `random.choices(results, k=len(results))` performs one bootstrap sample.


In [ ]:
# Exercise 1: fill in a bootstrap confidence interval

import random

def bootstrap_ci(results, n_boot=2000):
    """Return (lower bound, upper bound)."""
    stats = []
    for _ in range(n_boot):
        # TODO: Replace the triple-quoted content below with your code
        """Sample len(results) items with replacement and append the mean to stats."""
    stats.sort()
    return stats[int(0.025 * n_boot)], stats[int(0.975 * n_boot)]

random.seed(0)
lo, hi = bootstrap_ci([1] * 9 + [0] * 3)   # nine correct out of twelve
assert lo <= 0.75 <= hi
assert lo < 0.55 and hi > 0.9              # small samples produce appropriately wide uncertainty
print("Exercise 1 passed: a score needs an interval to be meaningful")


### Exercise 2: Write a More Robust Answer Parser

Outputs may contain commas or units, such as `1,234` or `42 tokens`. Remove commas and return the last number.

Hint: call `text.replace(",", "")`, then use `re.findall(r"\d+", ...)`.


In [ ]:
# Exercise 2: fill in an improved parser

import re

def parse_answer(text):
    """Extract the last integer after removing thousands separators; return None when absent."""
    # TODO: Replace the triple-quoted content below with your code
    """Remove commas, find all integers, convert the last to int, or return None if no match."""

assert parse_answer("The answer is: 1,234.") == 1234
assert parse_answer("used 42 tokens, cost 3 tokens") == 3
assert parse_answer("no number here") is None
print("Exercise 2 passed: the parser is the easiest part of an evaluation pipeline to break")


### Exercise 3: Quantify Position Bias

Judge both orders. A flip occurs when the two orderings select different underlying answers; the flip rate directly measures position sensitivity.

Hint: after the second judgment, map the selected position back to the answer identity with `3 - position`.


In [ ]:
# Exercise 3: fill in flip rate

import random

def flip_rate(judge, pairs=100):
    'Conclusion'
    flips = 0
    for _ in range(pairs):
        golden = random.choice([1, 2])
        picked1 = judge(golden)
        picked2 = 3 - judge(3 - golden)   # convert the swapped position back to answer ID
        # TODO: Replace the triple-quoted content below with your code
        """Increment flips when picked1 and picked2 disagree."""
    return flips / pairs

random.seed(0)
rate = flip_rate(lambda g: 1 if random.random() < 0.3 else g)
assert 0.2 < rate < 0.65, rate
print("Exercise 3 passed: you can audit a judge by reversing answer order")
